# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The baseline prioritizes content for human review using observed performance and available content signals. It uses simple, explicit rules rather than trained weights, and its score is a decision-support tool, not a prediction of search rankings.

LOW_VISIBILITY: eligible content has low observed impressions.

LOW_CLICKS: eligible content has impressions but relatively few clicks.

CONTENT_GAP: required signal data is missing, so the item needs inspection rather than being treated as a zero.

NO_CLEAR_TRIGGER: available signals do not trigger a review rule.

Only features available at the scoring time will be used. Product status flags and future performance windows will be excluded from scoring. Reason codes will explain why each item was selected.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find the repository root from the notebook's working directory
cwd = Path.cwd()
print("Current directory:", cwd)

# Locate the starter dataset if it exists in this environment
candidates = [
    cwd / "data/raw/content_refresh_anonymized.csv",
    cwd.parent / "data/raw/content_refresh_anonymized.csv",
    cwd.parent.parent / "data/raw/content_refresh_anonymized.csv",
]

data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    print("Dataset not found in the current environment.")
    print("We will connect/load the dataset in the next step.")
else:
    df = pd.read_csv(data_path)
    print("Dataset loaded:", data_path)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    display(df.head(5))

Current directory: /content
Dataset not found in the current environment.
We will connect/load the dataset in the next step.


In [2]:
!git clone https://github.com/vishalseries7030/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 379, done.
remote: Counting objects: 100% (379/379), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 379 (delta 222), reused 326 (delta 184), pack-reused 0 (from 0)
Receiving objects: 100% (379/379), 1.95 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (222/222), done.


In [3]:
from pathlib import Path

repo = Path("/content/flyrank-ml-internship")
print("Repository exists:", repo.exists())

for path in [
    repo / "data/raw/content_refresh_anonymized.csv",
    repo / "docs/data-dictionary.md",
    repo / "work/notebooks/w04_baseline_score.ipynb",
]:
    print(path, "->", path.exists())

Repository exists: True
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv -> True
/content/flyrank-ml-internship/docs/data-dictionary.md -> True
/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb -> True


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

repo = Path("/content/flyrank-ml-internship")
data_path = repo / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values — top 15 columns:")
display(df.isna().sum().sort_values(ascending=False).head(15))

Dataset loaded successfully
Rows: 30000
Columns: 44

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Missing values — top 15 columns:


,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468


In [5]:
# Create a separate working copy
baseline = df.copy()

# Explicit review rules — no trained weights
baseline["low_ctr_opportunity"] = (
    (baseline["impressions_90d"] >= 100)
    & (baseline["ctr"].notna())
    & (baseline["ctr"] < 1.0)
)

baseline["stale_content"] = (
    (baseline["content_age_days"] >= 180)
    & (baseline["days_since_last_update"] >= 90)
)

baseline["thin_content"] = (
    baseline["word_count"].notna()
    & (baseline["word_count"] < 1000)
)

baseline["missing_word_count"] = baseline["word_count"].isna()

# Transparent additive score: one point per triggered review rule
baseline["baseline_score"] = (
    baseline["low_ctr_opportunity"].astype(int)
    + baseline["stale_content"].astype(int)
    + baseline["thin_content"].astype(int)
    + baseline["missing_word_count"].astype(int)
)

# Explain the highest-priority trigger for each page
def assign_reason(row):
    reasons = []
    if row["low_ctr_opportunity"]:
        reasons.append("LOW_CTR_OPPORTUNITY")
    if row["stale_content"]:
        reasons.append("CONTENT_REFRESH")
    if row["thin_content"]:
        reasons.append("THIN_CONTENT")
    if row["missing_word_count"]:
        reasons.append("MISSING_CONTENT_DATA")
    return "|".join(reasons) if reasons else "NO_CLEAR_TRIGGER"

baseline["reason_code"] = baseline.apply(assign_reason, axis=1)

# Rank high scores first; IDs break ties for stable ordering only
baseline = baseline.sort_values(
    ["baseline_score", "content_id"],
    ascending=[False, True]
).reset_index(drop=True)

print("Rows:", len(baseline))
print("\nScore distribution:")
display(baseline["baseline_score"].value_counts().sort_index())

print("\nReason-code distribution:")
display(baseline["reason_code"].value_counts())

print("\nTop 20 review candidates:")
display(
    baseline[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "impressions_90d",
            "ctr",
            "content_age_days",
            "days_since_last_update",
            "word_count",
        ]
    ].head(20)
)

Rows: 30000

Score distribution:


,count
baseline_score,
0,5777
1,14273
2,7155
3,2795



Reason-code distribution:


,count
reason_code,
LOW_CTR_OPPORTUNITY,11499
NO_CLEAR_TRIGGER,5777
LOW_CTR_OPPORTUNITY|MISSING_CONTENT_DATA,3656
LOW_CTR_OPPORTUNITY|CONTENT_REFRESH,3183
LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CONTENT_DATA,2794
MISSING_CONTENT_DATA,1038
CONTENT_REFRESH,869
THIN_CONTENT,867
CONTENT_REFRESH|MISSING_CONTENT_DATA,211



Top 20 review candidates:


,content_id,baseline_score,reason_code,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_0003a66b9fa5,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,1940,0.26,329,104,NaN
1,content_00042377f909,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,441,0.45,256,104,NaN
2,content_001b45a25ada,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,2428,0.12,330,104,NaN
3,content_0062378f1bd9,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,1029,0.00,313,104,NaN
4,content_008976632003,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,9318,0.43,330,104,NaN
5,content_0089776ebb1f,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,3307,0.06,330,104,NaN
6,content_00a292fd4e2b,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,2612,0.15,313,104,NaN
7,content_00a44e45d37c,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,29692,0.03,330,104,NaN
8,content_00b0b3437a65,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,6321,0.27,236,104,NaN
9,content_00b1fc4e9026,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,6006,0.75,330,104,NaN


In [6]:
# Rank by score first, then by impressions for tie-breaking
baseline = baseline.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

# Add a practical human-review recommendation
def review_action(row):
    actions = []
    if row["low_ctr_opportunity"]:
        actions.append("Review search title and snippet")
    if row["stale_content"]:
        actions.append("Check whether content needs updating")
    if row["thin_content"]:
        actions.append("Review content depth")
    if row["missing_word_count"]:
        actions.append("Verify missing word-count data")
    return "; ".join(actions) if actions else "No immediate rule-based action"

def confidence_note(row):
    if row["missing_word_count"]:
        return "Medium-low: missing word-count data; verify before deciding"
    if row["low_ctr_opportunity"] and row["stale_content"]:
        return "Medium: multiple review signals; inspect page context"
    if row["low_ctr_opportunity"] or row["stale_content"] or row["thin_content"]:
        return "Medium-low: rule-based signal only; human review needed"
    return "Low: no explicit review trigger"

def what_could_make_it_wrong(row):
    issues = []
    if row["missing_word_count"]:
        issues.append("Word count is unavailable")
    if row["low_ctr_opportunity"]:
        issues.append("CTR may depend on query mix, position, and search intent")
    if row["stale_content"]:
        issues.append("Age alone does not prove the content is outdated")
    if row["thin_content"]:
        issues.append("Short content may still fully answer the user's need")
    return "; ".join(issues) if issues else "Available signals may not reveal the page's actual quality"

baseline["recommended_action"] = baseline.apply(review_action, axis=1)
baseline["confidence_note"] = baseline.apply(confidence_note, axis=1)
baseline["what_could_make_it_wrong"] = baseline.apply(
    what_could_make_it_wrong, axis=1
)

# Export without label-source columns or IDs as scoring features.
# IDs are retained only to identify pages for review.
output_columns = [
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_could_make_it_wrong",
    "impressions_90d",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

output_dir = repo / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
baseline[output_columns].to_csv(output_path, index=False)

print("Saved:", output_path)
print("Export shape:", baseline[output_columns].shape)

print("\nTop 20 review queue:")
display(baseline[output_columns].head(20))

Saved: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Export shape: (30000, 12)

Top 20 review queue:


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_5fe46e04994d,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,517715,0.14,537,104,NaN
1,content_2c2606c5d176,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,347399,0.53,362,104,NaN
2,content_9532f197bbc8,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,309192,0.87,445,104,NaN
3,content_c8e9d6ab9013,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,208678,0.00,362,104,NaN
4,content_d17681677e69,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,201584,0.24,313,104,NaN
5,content_a7427266c305,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,201111,0.11,257,104,NaN
6,content_01908772c6db,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,187893,0.45,313,104,NaN
7,content_bb5bd5f771dc,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,176296,0.23,236,104,NaN
8,content_91652435f57a,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,159590,0.06,257,104,NaN
9,content_3430a8b94511,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify be...,Word count is unavailable; CTR may depend on q...,152617,0.29,329,104,NaN


In [7]:
# Confirm the scoring rules do not use label-related fields
scoring_inputs = [
    "impressions_90d",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

forbidden = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "impressions_prev_30d",
}

print("Forbidden fields used in scoring:", forbidden.intersection(scoring_inputs))
print("IDs used as score features: No")
print("Scoring uses only explicit rule inputs:", scoring_inputs)

# Check systematic missingness by content type
print("\nWord-count missingness by content type:")
display(
    df.assign(word_count_missing=df["word_count"].isna())
      .groupby("content_type")["word_count_missing"]
      .agg(["count", "sum", "mean"])
      .sort_values("mean", ascending=False)
)

# Inspect top-20 picks for potential weak signals
top20 = baseline.head(20).copy()

print("\nTop-20 content types:")
display(top20["content_type"].value_counts(dropna=False))

print("\nTop-20 CTR and position context:")
display(
    top20[
        [
            "content_id",
            "content_type",
            "impressions_90d",
            "ctr",
            "avg_position",
            "word_count",
            "reason_code",
        ]
    ]
)

# Output integrity
print("\nCSV exists:", output_path.exists())
print("CSV rows:", pd.read_csv(output_path).shape[0])

Forbidden fields used in scoring: set()
IDs used as score features: No
Scoring uses only explicit rule inputs: ['impressions_90d', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count']

Word-count missingness by content type:


,count,sum,mean
content_type,,,
keyword article,27207,7699,0.282979
comparison article,697,0,0.000000
feedly article,2096,0,0.000000



Top-20 content types:


,count
content_type,
keyword article,20



Top-20 CTR and position context:


,content_id,content_type,impressions_90d,ctr,avg_position,word_count,reason_code
0,content_5fe46e04994d,keyword article,517715,0.14,4.2,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
1,content_2c2606c5d176,keyword article,347399,0.53,4.2,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
2,content_9532f197bbc8,keyword article,309192,0.87,2.0,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
3,content_c8e9d6ab9013,keyword article,208678,0.00,9.7,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
4,content_d17681677e69,keyword article,201584,0.24,5.8,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
5,content_a7427266c305,keyword article,201111,0.11,5.7,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
6,content_01908772c6db,keyword article,187893,0.45,4.0,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
7,content_bb5bd5f771dc,keyword article,176296,0.23,4.3,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
8,content_91652435f57a,keyword article,159590,0.06,7.8,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...
9,content_3430a8b94511,keyword article,152617,0.29,3.3,NaN,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...



CSV exists: True
CSV rows: 30000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Build the ranked queue
This baseline assigns one point for each triggered review rule: low CTR opportunity, stale content, thin content, and missing word-count data. Pages are ranked by score in descending order, then by 90-day impressions in descending order. Content ID is used only to break remaining ties consistently.
The ranked queue includes a reason code, recommended action, confidence note, and a description of what could make the recommendation wrong. The complete queue is exported to work/outputs/baseline_action_score.csv.
The score is a transparent review-priority heuristic, not a trained model or a prediction of future performance. Missing word-count data is flagged for verification, not treated as proof of thin content

In [8]:
# ML-07: Build and export the ranked baseline review queue

baseline = df.copy()

# Rule 1: Low CTR with sufficient search impressions
baseline["low_ctr_opportunity"] = (
    (baseline["impressions_90d"] >= 100)
    & baseline["ctr"].notna()
    & (baseline["ctr"] < 1.0)
)

# Rule 2: Older content that has not been updated recently
baseline["stale_content"] = (
    (baseline["content_age_days"] >= 180)
    & (baseline["days_since_last_update"] >= 90)
)

# Rule 3: Short content, only when word count is measured
baseline["thin_content"] = (
    baseline["word_count"].notna()
    & (baseline["word_count"] < 1000)
)

# Rule 4: Missing word count is a data-verification signal
baseline["missing_word_count"] = baseline["word_count"].isna()

# Transparent score: one point per triggered rule
baseline["baseline_score"] = (
    baseline["low_ctr_opportunity"].astype(int)
    + baseline["stale_content"].astype(int)
    + baseline["thin_content"].astype(int)
    + baseline["missing_word_count"].astype(int)
)

def assign_reason(row):
    reasons = []
    if row["low_ctr_opportunity"]:
        reasons.append("LOW_CTR_OPPORTUNITY")
    if row["stale_content"]:
        reasons.append("CONTENT_REFRESH")
    if row["thin_content"]:
        reasons.append("THIN_CONTENT")
    if row["missing_word_count"]:
        reasons.append("MISSING_CONTENT_DATA")
    return "|".join(reasons) if reasons else "NO_CLEAR_TRIGGER"

baseline["reason_code"] = baseline.apply(assign_reason, axis=1)

def review_action(row):
    actions = []
    if row["low_ctr_opportunity"]:
        actions.append("Review search title and snippet")
    if row["stale_content"]:
        actions.append("Check whether content needs updating")
    if row["thin_content"]:
        actions.append("Review content depth")
    if row["missing_word_count"]:
        actions.append("Verify missing word-count data")
    return "; ".join(actions) if actions else "No immediate rule-based action"

def confidence_note(row):
    if row["missing_word_count"]:
        return "Medium-low: missing word-count data; verify first"
    if row["low_ctr_opportunity"] and row["stale_content"]:
        return "Medium: multiple signals; inspect page context"
    if row["low_ctr_opportunity"] or row["stale_content"] or row["thin_content"]:
        return "Medium-low: rule-based signal; human review needed"
    return "Low: no explicit review trigger"

def what_could_make_it_wrong(row):
    issues = []
    if row["missing_word_count"]:
        issues.append("Word count is unavailable")
    if row["low_ctr_opportunity"]:
        issues.append("CTR depends on query mix, position, and intent")
    if row["stale_content"]:
        issues.append("Age alone does not prove content is outdated")
    if row["thin_content"]:
        issues.append("Short content may still meet user needs")
    return "; ".join(issues) if issues else "Available signals may not reveal actual page quality"

baseline["recommended_action"] = baseline.apply(review_action, axis=1)
baseline["confidence_note"] = baseline.apply(confidence_note, axis=1)
baseline["what_could_make_it_wrong"] = baseline.apply(
    what_could_make_it_wrong, axis=1
)

# Stable ranking: score, impressions, then ID
baseline = baseline.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

# Export the full ranked queue
output_columns = [
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_could_make_it_wrong",
    "impressions_90d",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

output_dir = repo / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
baseline[output_columns].to_csv(output_path, index=False)

print("Rows ranked:", len(baseline))
print("CSV saved:", output_path)
print("Export shape:", baseline[output_columns].shape)
display(baseline[output_columns].head(20))

Rows ranked: 30000
CSV saved: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Export shape: (30000, 12)


,content_id,client_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,content_age_days,days_since_last_update,word_count
0,content_5fe46e04994d,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,517715,0.14,537,104,NaN
1,content_2c2606c5d176,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,347399,0.53,362,104,NaN
2,content_9532f197bbc8,client_4e07408562,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,309192,0.87,445,104,NaN
3,content_c8e9d6ab9013,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,208678,0.00,362,104,NaN
4,content_d17681677e69,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201584,0.24,313,104,NaN
5,content_a7427266c305,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201111,0.11,257,104,NaN
6,content_01908772c6db,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,187893,0.45,313,104,NaN
7,content_bb5bd5f771dc,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,176296,0.23,236,104,NaN
8,content_91652435f57a,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,159590,0.06,257,104,NaN
9,content_3430a8b94511,client_19581e27de,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,152617,0.29,329,104,NaN


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review

The top-20 queue prioritizes pages with the highest baseline score. Ties are broken by 90-day impressions in descending order, with content ID used only for stable ordering.

Review approach

Action: Review search titles and snippets for pages flagged with low CTR; check update needs for older pages; verify missing word-count data before making content-depth decisions.

Reason code: Each page carries the rule or combination of rules that triggered its selection.

Confidence: These are rule-based review recommendations, not predictions. Human inspection is required before changes are made.

What could make the recommendation wrong: CTR can vary with query intent and position; age alone does not prove content is stale; missing word count does not mean the page is thin.

The queue is a prioritization aid, not evidence that changing a page will improve performance.

In [9]:
# ML-07: Top-20 review table
top20_review = baseline.head(20).copy()

review_columns = [
    "content_id",
    "baseline_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_could_make_it_wrong",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

# Verify the review table has 20 rows and all required review fields
assert len(top20_review) == 20
assert top20_review["recommended_action"].notna().all()
assert top20_review["reason_code"].notna().all()
assert top20_review["confidence_note"].notna().all()
assert top20_review["what_could_make_it_wrong"].notna().all()

print("Top-20 review ready:", len(top20_review), "pages")
display(top20_review[review_columns])

# Inspect a potential weak pick:
# a high score is not proof that a page should definitely be changed.
weak_pick = top20_review.iloc[0]

print("\nWeak-pick review")
print("Content ID:", weak_pick["content_id"])
print("Score:", weak_pick["baseline_score"])
print("Reason:", weak_pick["reason_code"])
print("Why this might be a weak pick:", weak_pick["what_could_make_it_wrong"])

Top-20 review ready: 20 pages


,content_id,baseline_score,reason_code,recommended_action,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count
0,content_5fe46e04994d,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,517715,0.14,4.2,537,104,NaN
1,content_2c2606c5d176,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,347399,0.53,4.2,362,104,NaN
2,content_9532f197bbc8,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,309192,0.87,2.0,445,104,NaN
3,content_c8e9d6ab9013,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,208678,0.00,9.7,362,104,NaN
4,content_d17681677e69,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201584,0.24,5.8,313,104,NaN
5,content_a7427266c305,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,201111,0.11,5.7,257,104,NaN
6,content_01908772c6db,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,187893,0.45,4.0,313,104,NaN
7,content_bb5bd5f771dc,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,176296,0.23,4.3,236,104,NaN
8,content_91652435f57a,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,159590,0.06,7.8,257,104,NaN
9,content_3430a8b94511,3,LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CO...,Review search title and snippet; Check whether...,Medium-low: missing word-count data; verify first,Word count is unavailable; CTR depends on quer...,152617,0.29,3.3,329,104,NaN



Weak-pick review
Content ID: content_5fe46e04994d
Score: 3
Reason: LOW_CTR_OPPORTUNITY|CONTENT_REFRESH|MISSING_CONTENT_DATA
Why this might be a weak pick: Word count is unavailable; CTR depends on query mix, position, and intent; Age alone does not prove content is outdated


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks and leakage check

Weak picks

A page can rank highly because several simple rules are triggered, but this does not prove that it is the best page to update first.

Low CTR alone can be misleading because search intent, query mix, and average position affect click-through rate.

Older content is not necessarily outdated. A human reviewer should confirm whether the information needs updating.

Missing word count is concentrated in keyword articles: 7,699 of 27,207 keyword articles have missing word counts, while comparison and Feedly articles have no missing word counts in this dataset. This is systematic missingness, so it must not be interpreted as proof of thin content.

The top-20 queue contains only keyword articles. This is a limitation worth monitoring; the baseline is a heuristic, not a fair or validated allocation of review resources.

Leakage check

The score uses only impressions_90d, ctr, content_age_days, days_since_last_update, and the presence or absence of word_count. It does not use trend_direction, trend_pct, is_declining_label, IDs, or the 30-day comparison windows to calculate the score. IDs are retained only to identify records and break ties consistently.

This is a rule-based prioritization baseline, not a trained predictive model. No precision-at-20 claim is made because this workflow has not evaluated the ranked queue against an independently verified outcome label.

In [10]:
# ML-07 final validation
saved = pd.read_csv(output_path)

assert len(saved) == len(df) == 30000
assert saved["content_id"].is_unique
assert saved["baseline_score"].notna().all()
assert saved["reason_code"].notna().all()

for col in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "impressions_prev_30d",
]:
    assert col not in saved.columns, f"Potential leakage field: {col}"

print("PASS: 30,000 rows exported")
print("PASS: Content IDs are unique")
print("PASS: Scores and reason codes are present")
print("PASS: Leakage-related fields excluded from CSV")
print("CSV saved at:", output_path)

display(saved["baseline_score"].value_counts().sort_index())

PASS: 30,000 rows exported
PASS: Content IDs are unique
PASS: Scores and reason codes are present
PASS: Leakage-related fields excluded from CSV
CSV saved at: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


,count
baseline_score,
0,5777
1,14273
2,7155
3,2795


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.